# Risk Condition Benchmark

Which aggregation condition is the most reliable for detecting shortcuts?

We stress-test all 5 conditions using **all 6 detection methods** across **easy, medium, and hard** synthetic scenarios, measure false-positive / false-negative rates, and identify the best default.

**Methods used:** probe, statistical, hbac, geometric, frequency, gce

**Figures saved to:** `output/risk-recommendations/`

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 11, "figure.dpi": 120})
import os, warnings, logging
warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)

OUT = "output/risk-recommendations"
os.makedirs(OUT, exist_ok=True)

from shortcut_detect import ShortcutDetector, generate_linear_shortcut, generate_no_shortcut

CONDITIONS = ["indicator_count", "majority_vote", "weighted_risk", "multi_attribute", "meta_classifier"]
COND_SHORT = ["ind_count", "maj_vote", "weighted", "multi_attr", "meta_clf"]
RISK_COLORS = {"HIGH": "#e74c3c", "MODERATE": "#f39c12", "LOW": "#27ae60"}
METHODS = ["probe", "statistical", "hbac", "geometric", "frequency", "gce"]
COLORS = ["#3498db", "#2ecc71", "#e67e22", "#9b59b6", "#e74c3c"]

def make_correlated_groups(labels, seed, flip_rate=0.1):
    rng = np.random.RandomState(seed)
    grp = labels.copy()
    flip_mask = rng.rand(len(labels)) < flip_rate
    grp[flip_mask] = 1 - grp[flip_mask]
    return grp

def classify_risk(text):
    t = text.upper()
    if "HIGH RISK" in t: return "HIGH"
    if "MODERATE RISK" in t: return "MODERATE"
    return "LOW"

def risk_to_num(r):
    return {"LOW": 0, "MODERATE": 1, "HIGH": 2}[r]

def benchmark_conditions(emb, grp):
    import io, contextlib
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        base = ShortcutDetector(methods=METHODS)
        base.fit(emb, grp)
    verdicts = {}
    for cond in CONDITIONS:
        det = ShortcutDetector(methods=METHODS, condition_name=cond)
        det.results_ = base.results_
        det.methods = METHODS
        verdicts[cond] = classify_risk(det._generate_overall_assessment())
    return verdicts

print(f"Benchmarking {len(CONDITIONS)} conditions with {len(METHODS)} methods: {METHODS}")

Benchmarking 5 conditions with 6 methods: ['probe', 'statistical', 'hbac', 'geometric', 'frequency', 'gce']


---
## Figure 1 — Detection Accuracy Across Difficulty Levels

For each difficulty (shortcut dims + sample size), run 20 seeds.
A correct result: shortcut data flagged as MODERATE/HIGH, clean data flagged as LOW.

In [2]:
difficulties = [
    ("Hard\n(1 dim, n=150)",   1, 150, 0.15),
    ("Medium\n(2 dims, n=200)", 2, 200, 0.12),
    ("Easy\n(5 dims, n=300)",   5, 300, 0.10),
]
n_seeds = 20

accuracy = {c: [] for c in CONDITIONS}

for label, sd, n, flip in difficulties:
    correct = {c: 0 for c in CONDITIONS}
    total = 2 * n_seeds
    for seed in range(n_seeds):
        # Biased: correct if not LOW
        emb, lab = generate_linear_shortcut(n_samples=n, embedding_dim=64, shortcut_dims=sd, seed=seed)
        grp = make_correlated_groups(lab, seed=seed, flip_rate=flip)
        for c, v in benchmark_conditions(emb, grp).items():
            if v != "LOW": correct[c] += 1
        # Clean: correct if LOW
        emb, lab = generate_no_shortcut(n_samples=n, embedding_dim=64, seed=seed)
        grp = np.random.RandomState(seed).randint(0, 2, size=len(lab))
        for c, v in benchmark_conditions(emb, grp).items():
            if v == "LOW": correct[c] += 1
    for c in CONDITIONS:
        accuracy[c].append(correct[c] / total)
    print(f"  {label.split(chr(10))[0]} done")

# Plot
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(difficulties))
w = 0.15

for i, c in enumerate(CONDITIONS):
    bars = ax.bar(x + i * w - 2 * w, accuracy[c], w, label=COND_SHORT[i], color=COLORS[i], edgecolor="white")
    for j, v in enumerate(accuracy[c]):
        ax.text(x[j] + i * w - 2 * w, v + 0.01, f"{v:.0%}", ha="center", fontsize=8, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels([d[0] for d in difficulties])
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.15)
ax.set_title("Detection Accuracy by Difficulty (20 seeds each)")
ax.legend(loc="lower right", fontsize=9)
ax.axhline(0.5, color="gray", ls="--", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT}/fig1_accuracy_by_difficulty.png", dpi=150, bbox_inches="tight")
plt.show()

  Hard done


  Medium done


  Easy done


---
## Figure 2 — False Positive vs False Negative Rates (Hard Scenario)

The hard scenario (2 shortcut dims, n=150) is where conditions diverge.
Low FP = fewer false alarms. Low FN = catches weak shortcuts.

In [3]:
n_seeds = 30
fp = {c: 0 for c in CONDITIONS}
fn = {c: 0 for c in CONDITIONS}

for seed in range(n_seeds):
    # Biased (2 dims, n=150) -> should detect
    emb, lab = generate_linear_shortcut(n_samples=150, embedding_dim=64, shortcut_dims=2, seed=seed)
    grp = make_correlated_groups(lab, seed=seed, flip_rate=0.15)
    for c, v in benchmark_conditions(emb, grp).items():
        if v == "LOW": fn[c] += 1
    # Clean -> should not detect
    emb, lab = generate_no_shortcut(n_samples=150, embedding_dim=64, seed=seed)
    grp = np.random.RandomState(seed).randint(0, 2, size=len(lab))
    for c, v in benchmark_conditions(emb, grp).items():
        if v != "LOW": fp[c] += 1

fp_fn = pd.DataFrame({
    "FP Rate": {COND_SHORT[i]: fp[c] / n_seeds for i, c in enumerate(CONDITIONS)},
    "FN Rate": {COND_SHORT[i]: fn[c] / n_seeds for i, c in enumerate(CONDITIONS)},
})
fp_fn["Total Error"] = (fp_fn["FP Rate"] + fp_fn["FN Rate"]) / 2

fig, ax = plt.subplots(figsize=(9, 4.5))
x_pos = np.arange(len(CONDITIONS))
w = 0.3
ax.bar(x_pos - w/2, fp_fn["FP Rate"], w, label="False Positive (false alarm)", color="#e74c3c", alpha=0.85)
ax.bar(x_pos + w/2, fp_fn["FN Rate"], w, label="False Negative (missed shortcut)", color="#3498db", alpha=0.85)
ax.set_xticks(x_pos)
ax.set_xticklabels(COND_SHORT)
ax.set_ylabel("Rate")
ax.set_title("FP / FN Rates on Hard Scenario (2 dims, n=150, 30 seeds)")
ax.legend(fontsize=9)
for i in range(len(CONDITIONS)):
    err = fp_fn.iloc[i]["Total Error"]
    top = max(fp_fn.iloc[i]["FP Rate"], fp_fn.iloc[i]["FN Rate"])
    ax.text(i, top + 0.01, f"err={err:.0%}", ha="center", fontsize=9, fontweight="bold")
ax.set_ylim(0, max(fp_fn[["FP Rate", "FN Rate"]].max()) + 0.08)
plt.tight_layout()
plt.savefig(f"{OUT}/fig2_fp_fn_hard.png", dpi=150, bbox_inches="tight")
plt.show()
print(fp_fn.to_string())

             FP Rate  FN Rate  Total Error
ind_count   1.000000      0.0     0.500000
maj_vote    1.000000      0.0     0.500000
weighted    0.533333      0.0     0.266667
multi_attr  1.000000      0.0     0.500000
meta_clf    0.033333      0.0     0.016667


---
## Figure 3 — Sensitivity Curve Across Shortcut Strength

Detection rate (true positive) as shortcut dims increase from 0 to 10.
At 0 dims (no shortcut), detection should be 0 (no false alarms). At 1+ dims, should approach 1.

In [4]:
dims_range = [0, 1, 2, 3, 5, 8, 10]
n_seeds = 15
detection_rate = {c: [] for c in CONDITIONS}

for sd in dims_range:
    detected = {c: 0 for c in CONDITIONS}
    for seed in range(n_seeds):
        if sd == 0:
            emb, lab = generate_no_shortcut(n_samples=200, embedding_dim=64, seed=seed)
            grp = np.random.RandomState(seed).randint(0, 2, size=len(lab))
        else:
            emb, lab = generate_linear_shortcut(n_samples=200, embedding_dim=64, shortcut_dims=sd, seed=seed)
            grp = make_correlated_groups(lab, seed=seed, flip_rate=0.12)
        for c, v in benchmark_conditions(emb, grp).items():
            if v != "LOW": detected[c] += 1
    for c in CONDITIONS:
        detection_rate[c].append(detected[c] / n_seeds)
    print(f"  dims={sd} done")

fig, ax = plt.subplots(figsize=(9, 5))
markers = ["o", "s", "D", "^", "*"]
for i, c in enumerate(CONDITIONS):
    ax.plot(dims_range, detection_rate[c], marker=markers[i], label=COND_SHORT[i],
            color=COLORS[i], linewidth=2, markersize=7)

ax.axhspan(0, 0.05, color="green", alpha=0.07)
ax.text(0.3, 0.02, "ideal at 0 dims (no false positives)", fontsize=8, color="gray")
ax.text(6, 0.82, "ideal at 1+ dims (detect all)", fontsize=8, color="gray")
ax.set_xlabel("Shortcut Dimensions")
ax.set_ylabel("Detection Rate (flagged MODERATE or HIGH)")
ax.set_title("Sensitivity Curve: Detection Rate vs Shortcut Strength (15 seeds)")
ax.legend(loc="center right", fontsize=9)
ax.set_ylim(-0.05, 1.1)
ax.set_xticks(dims_range)
plt.tight_layout()
plt.savefig(f"{OUT}/fig3_sensitivity_curve.png", dpi=150, bbox_inches="tight")
plt.show()

  dims=0 done


  dims=1 done


  dims=2 done


  dims=3 done


  dims=5 done


  dims=8 done


  dims=10 done


---
## Figure 4 — Risk Level Distribution by Scenario

For each condition, what fraction of runs produce LOW / MODERATE / HIGH
across clean vs biased data? Shows calibration quality.

In [5]:
n_seeds = 20
scen_defs = [
    ("Clean\n(no shortcut)", 0, 200, 0),
    ("Weak\n(1 dim)",        1, 200, 0.15),
    ("Medium\n(3 dims)",     3, 200, 0.10),
    ("Strong\n(5 dims)",     5, 200, 0.10),
]

counts = {}
for scen_name, sd, n, flip in scen_defs:
    for c in CONDITIONS:
        counts[(scen_name, c)] = {"LOW": 0, "MODERATE": 0, "HIGH": 0}
    for seed in range(n_seeds):
        if sd == 0:
            emb, lab = generate_no_shortcut(n_samples=n, embedding_dim=64, seed=seed)
            grp = np.random.RandomState(seed).randint(0, 2, size=len(lab))
        else:
            emb, lab = generate_linear_shortcut(n_samples=n, embedding_dim=64, shortcut_dims=sd, seed=seed)
            grp = make_correlated_groups(lab, seed=seed, flip_rate=flip)
        for c, v in benchmark_conditions(emb, grp).items():
            counts[(scen_name, c)][v] += 1

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5), sharey=True)
for ax_idx, (scen_name, _, _, _) in enumerate(scen_defs):
    ax = axes[ax_idx]
    low_vals  = [counts[(scen_name, c)]["LOW"] / n_seeds for c in CONDITIONS]
    mod_vals  = [counts[(scen_name, c)]["MODERATE"] / n_seeds for c in CONDITIONS]
    high_vals = [counts[(scen_name, c)]["HIGH"] / n_seeds for c in CONDITIONS]
    x_pos = np.arange(len(CONDITIONS))
    ax.bar(x_pos, low_vals, 0.6, label="LOW", color=RISK_COLORS["LOW"])
    ax.bar(x_pos, mod_vals, 0.6, bottom=low_vals, label="MODERATE", color=RISK_COLORS["MODERATE"])
    ax.bar(x_pos, high_vals, 0.6, bottom=[l+m for l,m in zip(low_vals, mod_vals)], label="HIGH", color=RISK_COLORS["HIGH"])
    ax.set_xticks(x_pos)
    ax.set_xticklabels(COND_SHORT, rotation=35, ha="right", fontsize=8)
    ax.set_title(scen_name, fontsize=10, fontweight="bold")
    ax.set_ylim(0, 1.05)
    if ax_idx == 0:
        ax.set_ylabel("Fraction of runs")

axes[-1].legend(loc="center right", fontsize=8, bbox_to_anchor=(1.35, 0.5))
fig.suptitle("Risk Level Distribution by Scenario (20 seeds each)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUT}/fig4_risk_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Figure 5 — Overall Ranking

Composite score: average accuracy across all difficulty levels, weighted
toward hard scenarios (where conditions actually differ).

In [6]:
# Weighted composite: hard=3x, medium=2x, easy=1x
weights = [3, 2, 1]
composite = {}
for i, c in enumerate(CONDITIONS):
    score = sum(w * a for w, a in zip(weights, accuracy[c])) / sum(weights)
    composite[COND_SHORT[i]] = score

ranking = sorted(composite.items(), key=lambda x: x[1], reverse=True)
names  = [r[0] for r in ranking]
scores = [r[1] for r in ranking]

fig, ax = plt.subplots(figsize=(9, 4.5))
bar_colors = ["#2ecc71" if s == max(scores) else "#3498db" for s in scores]
bars = ax.barh(names[::-1], scores[::-1], color=bar_colors[::-1], edgecolor="white", height=0.6)

for i, (n, s) in enumerate(zip(names[::-1], scores[::-1])):
    ax.text(s + 0.005, i, f"{s:.1%}", va="center", fontsize=11, fontweight="bold")

ax.set_xlabel("Weighted Accuracy (hard=3x, medium=2x, easy=1x)")
ax.set_title("Overall Condition Ranking")
ax.set_xlim(0, 1.12)
ax.axvline(1.0, color="gray", ls="--", alpha=0.3)

best = ranking[0]
ax.annotate(f"Best: {best[0]} ({best[1]:.1%})",
            xy=(best[1], len(ranking)-1), xytext=(best[1]-0.15, len(ranking)-0.5),
            fontsize=10, fontweight="bold", color="#27ae60",
            arrowprops=dict(arrowstyle="->", color="#27ae60"))

plt.tight_layout()
plt.savefig(f"{OUT}/fig5_overall_ranking.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n=== Final Ranking ===")
for i, (n, s) in enumerate(ranking, 1):
    print(f"  {i}. {n:12s}  {s:.1%}")


=== Final Ranking ===
  1. meta_clf      97.9%
  2. weighted      71.7%
  3. ind_count     50.0%
  4. maj_vote      50.0%
  5. multi_attr    50.0%


---
## Figure 5b — Per-Method Benchmark: Individual Detection Method Accuracy

How accurate is each detection method **on its own**?
We run each method individually across the same difficulty levels and measure
whether it correctly flags biased data (MODERATE/HIGH) and passes clean data (LOW).

In [ ]:
import io, contextlib

METHOD_COLORS = ["#3498db", "#2ecc71", "#e67e22", "#9b59b6", "#e74c3c", "#1abc9c"]

def benchmark_single_method(emb, grp, method):
    """Run a single detection method and return its risk verdict."""
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        det = ShortcutDetector(methods=[method])
        det.fit(emb, grp)
    r = det.results_.get(method, {})
    risk = r.get("risk_value", "low") or "low"
    return {"low": "LOW", "moderate": "MODERATE", "high": "HIGH"}[risk]

# --- Part A: Per-method accuracy across difficulties ---
method_accuracy = {m: [] for m in METHODS}

for label, sd, n, flip in difficulties:
    correct = {m: 0 for m in METHODS}
    total = 2 * n_seeds
    for seed in range(n_seeds):
        # Biased: correct if not LOW
        emb, lab = generate_linear_shortcut(n_samples=n, embedding_dim=64, shortcut_dims=sd, seed=seed)
        grp = make_correlated_groups(lab, seed=seed, flip_rate=flip)
        for m in METHODS:
            if benchmark_single_method(emb, grp, m) != "LOW":
                correct[m] += 1
        # Clean: correct if LOW
        emb, lab = generate_no_shortcut(n_samples=n, embedding_dim=64, seed=seed)
        grp = np.random.RandomState(seed).randint(0, 2, size=len(lab))
        for m in METHODS:
            if benchmark_single_method(emb, grp, m) == "LOW":
                correct[m] += 1
    for m in METHODS:
        method_accuracy[m].append(correct[m] / total)
    print(f"  {label.split(chr(10))[0]} done")

# Plot accuracy
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(difficulties))
w = 0.12

for i, m in enumerate(METHODS):
    bars = ax.bar(x + i * w - 2.5 * w, method_accuracy[m], w,
                  label=m, color=METHOD_COLORS[i], edgecolor="white")
    for j, v in enumerate(method_accuracy[m]):
        ax.text(x[j] + i * w - 2.5 * w, v + 0.01, f"{v:.0%}",
                ha="center", fontsize=7, fontweight="bold", rotation=45)

ax.set_xticks(x)
ax.set_xticklabels([d[0] for d in difficulties])
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.2)
ax.set_title("Per-Method Detection Accuracy by Difficulty (20 seeds each)")
ax.legend(loc="lower right", fontsize=8, ncol=2)
ax.axhline(0.5, color="gray", ls="--", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT}/fig5b_method_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Part B: Per-method FP/FN on hard scenario ---
n_seeds_fp = 30
m_fp = {m: 0 for m in METHODS}
m_fn = {m: 0 for m in METHODS}

for seed in range(n_seeds_fp):
    # Biased -> should detect
    emb, lab = generate_linear_shortcut(n_samples=150, embedding_dim=64, shortcut_dims=2, seed=seed)
    grp = make_correlated_groups(lab, seed=seed, flip_rate=0.15)
    for m in METHODS:
        if benchmark_single_method(emb, grp, m) == "LOW":
            m_fn[m] += 1
    # Clean -> should not detect
    emb, lab = generate_no_shortcut(n_samples=150, embedding_dim=64, seed=seed)
    grp = np.random.RandomState(seed).randint(0, 2, size=len(lab))
    for m in METHODS:
        if benchmark_single_method(emb, grp, m) != "LOW":
            m_fp[m] += 1

method_fp_fn = pd.DataFrame({
    "FP Rate": {m: m_fp[m] / n_seeds_fp for m in METHODS},
    "FN Rate": {m: m_fn[m] / n_seeds_fp for m in METHODS},
})
method_fp_fn["Total Error"] = (method_fp_fn["FP Rate"] + method_fp_fn["FN Rate"]) / 2

fig, ax = plt.subplots(figsize=(10, 4.5))
x_pos = np.arange(len(METHODS))
w = 0.3
ax.bar(x_pos - w/2, method_fp_fn["FP Rate"], w,
       label="False Positive (false alarm)", color="#e74c3c", alpha=0.85)
ax.bar(x_pos + w/2, method_fp_fn["FN Rate"], w,
       label="False Negative (missed shortcut)", color="#3498db", alpha=0.85)
ax.set_xticks(x_pos)
ax.set_xticklabels(METHODS)
ax.set_ylabel("Rate")
ax.set_title("Per-Method FP / FN Rates on Hard Scenario (2 dims, n=150, 30 seeds)")
ax.legend(fontsize=9)
for i, m in enumerate(METHODS):
    err = method_fp_fn.loc[m, "Total Error"]
    top = max(method_fp_fn.loc[m, "FP Rate"], method_fp_fn.loc[m, "FN Rate"])
    ax.text(i, top + 0.01, f"err={err:.0%}", ha="center", fontsize=9, fontweight="bold")
ax.set_ylim(0, max(method_fp_fn[["FP Rate", "FN Rate"]].max()) + 0.1)
plt.tight_layout()
plt.savefig(f"{OUT}/fig5b_method_fp_fn.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Part C: Weighted ranking ---
weights = [3, 2, 1]
method_composite = {}
for m in METHODS:
    score = sum(w * a for w, a in zip(weights, method_accuracy[m])) / sum(weights)
    method_composite[m] = score

m_ranking = sorted(method_composite.items(), key=lambda x: x[1], reverse=True)
print("\n=== Per-Method Ranking (weighted accuracy) ===")
for i, (m, s) in enumerate(m_ranking, 1):
    fp_r = method_fp_fn.loc[m, "FP Rate"]
    fn_r = method_fp_fn.loc[m, "FN Rate"]
    print(f"  {i}. {m:15s}  acc={s:.1%}  FP={fp_r:.0%}  FN={fn_r:.0%}")

print("\n--- Comparison: best individual method vs meta_classifier condition ---")
best_m = m_ranking[0]
print(f"  Best method:      {best_m[0]:15s}  {best_m[1]:.1%}")
print(f"  Meta-classifier:  {'(ensemble)':15s}  {composite.get('meta_clf', 0):.1%}")
print(f"  Gap: meta_clf is {composite.get('meta_clf', 0) - best_m[1]:+.1%} over best single method")

---
## Figure 6 — Real Data: CheXpert Embeddings

How do the conditions behave on **real medical imaging data** where we don't have ground truth?
This shows the sensitivity vs specificity tradeoff: aggressive conditions (ind_count, maj_vote)
flag risk whenever *any* method disagrees, while conservative conditions (meta_clf) require
agreement from the most validated methods.

In [7]:
import io, contextlib

# Load CheXpert real data
data_dir = "../data"
for candidate in ["../data", "../../data", "data"]:
    if os.path.exists(os.path.join(candidate, "chest_embeddings.npy")):
        data_dir = candidate
        break

real_emb = np.load(os.path.join(data_dir, "chest_embeddings.npy"))
real_groups = np.load(os.path.join(data_dir, "chest_group_labels.npy"))
print(f"CheXpert: {real_emb.shape[0]} samples, {len(np.unique(real_groups))} groups")

# Run all methods
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    base = ShortcutDetector(methods=METHODS)
    base.fit(real_emb, real_groups)

# Per-method results
method_risks = {}
for m in METHODS:
    r = base.results_.get(m, {})
    method_risks[m] = r.get("risk_value", "low") or "low"
print("\nPer-method risk:")
for m, r in method_risks.items():
    print(f"  {m:15s} -> {r}")

# Per-condition verdicts
real_verdicts = {}
for cond in CONDITIONS:
    det = ShortcutDetector(methods=METHODS, condition_name=cond)
    det.results_ = base.results_
    det.methods = METHODS
    real_verdicts[cond] = classify_risk(det._generate_overall_assessment())

# --- Figure 6: two-panel ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={"width_ratios": [1, 1.2]})

# Panel A: per-method risk
risk_map = {"low": 0, "moderate": 1, "high": 2}
m_names = METHODS
m_vals = [risk_map.get(method_risks[m], 0) for m in m_names]
m_colors = [RISK_COLORS[{"low": "LOW", "moderate": "MODERATE", "high": "HIGH"}[method_risks[m]]] for m in m_names]
ax1.barh(m_names, m_vals, color=m_colors, edgecolor="white", height=0.6)
ax1.set_xticks([0, 1, 2])
ax1.set_xticklabels(["LOW", "MODERATE", "HIGH"])
for i, m in enumerate(m_names):
    ax1.text(m_vals[i] + 0.05, i, method_risks[m].upper(), va="center", fontsize=9, fontweight="bold")
ax1.set_xlim(-0.2, 2.8)
ax1.set_title("A) Per-Method Risk on CheXpert", fontweight="bold")

# Panel B: per-condition verdict
c_vals = [risk_to_num(real_verdicts[c]) for c in CONDITIONS]
c_colors = [RISK_COLORS[real_verdicts[c]] for c in CONDITIONS]
ax2.barh(COND_SHORT, c_vals, color=c_colors, edgecolor="white", height=0.6)
ax2.set_xticks([0, 1, 2])
ax2.set_xticklabels(["LOW", "MODERATE", "HIGH"])
for i, c in enumerate(CONDITIONS):
    ax2.text(c_vals[i] + 0.05, i, real_verdicts[c], va="center", fontsize=9, fontweight="bold")
ax2.set_xlim(-0.2, 2.8)
ax2.set_title("B) Per-Condition Verdict on CheXpert", fontweight="bold")

fig.suptitle("CheXpert Real Data: Methods Disagree, Conditions Diverge", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUT}/fig6_chexpert_real.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nKey insight: probe, statistical, hbac (validated methods) say LOW.")
print("frequency, gce, geometric (noisier methods) say HIGH/MODERATE.")
print("Aggressive conditions (ind_count, maj_vote) flag HIGH from any indicator.")
print("meta_classifier trusts the validated methods more -> LOW.")

CheXpert: 2000 samples, 4 groups



Per-method risk:
  probe           -> low
  statistical     -> low
  hbac            -> low
  geometric       -> moderate
  frequency       -> high
  gce             -> high

Key insight: probe, statistical, hbac (validated methods) say LOW.
frequency, gce, geometric (noisier methods) say HIGH/MODERATE.
Aggressive conditions (ind_count, maj_vote) flag HIGH from any indicator.
meta_classifier trusts the validated methods more -> LOW.


---
## Figure 7 — Real Data Benchmark: Per-Method Stability on CheXpert

Since we lack ground truth on real data, we benchmark **stability**: how consistent is each
method's verdict across 20 bootstrap resamples of the CheXpert embeddings?
Unstable methods flip between risk levels on slight data perturbations -- a sign of unreliability.

We also show per-method **indicator counts** and **numeric scores** to quantify each method's
confidence on real data, and compare individual methods against the meta-classifier ensemble.

In [ ]:
n_boot = 20
boot_verdicts = {m: [] for m in METHODS}
boot_cond_verdicts = {c: [] for c in CONDITIONS}

for b in range(n_boot):
    rng = np.random.RandomState(b)
    idx = rng.choice(len(real_emb), size=len(real_emb), replace=True)
    b_emb, b_grp = real_emb[idx], real_groups[idx]

    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        det = ShortcutDetector(methods=METHODS)
        det.fit(b_emb, b_grp)

    for m in METHODS:
        r = det.results_.get(m, {})
        risk = r.get("risk_value", "low") or "low"
        boot_verdicts[m].append({"low": "LOW", "moderate": "MODERATE", "high": "HIGH"}[risk])

    for cond in CONDITIONS:
        det2 = ShortcutDetector(methods=METHODS, condition_name=cond)
        det2.results_ = det.results_
        det2.methods = METHODS
        boot_cond_verdicts[cond].append(classify_risk(det2._generate_overall_assessment()))

print(f"Bootstrap done: {n_boot} resamples of {len(real_emb)} CheXpert samples")

# --- Panel A: method stability heatmap ---
risk_levels = ["LOW", "MODERATE", "HIGH"]
stability_data = []
for m in METHODS:
    row = [boot_verdicts[m].count(r) / n_boot for r in risk_levels]
    stability_data.append(row)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [1, 1]})

# Heatmap for methods
im = ax1.imshow(stability_data, cmap="RdYlGn_r", aspect="auto", vmin=0, vmax=1)
ax1.set_xticks(range(len(risk_levels)))
ax1.set_xticklabels(risk_levels)
ax1.set_yticks(range(len(METHODS)))
ax1.set_yticklabels(METHODS)
for i in range(len(METHODS)):
    for j in range(len(risk_levels)):
        val = stability_data[i][j]
        if val > 0:
            ax1.text(j, i, f"{val:.0%}", ha="center", va="center",
                     fontsize=10, fontweight="bold",
                     color="white" if val > 0.6 else "black")
ax1.set_title("A) Per-Method Verdict Stability\n(fraction of bootstrap resamples)", fontweight="bold")

# Heatmap for conditions
cond_stability = []
for c in CONDITIONS:
    row = [boot_cond_verdicts[c].count(r) / n_boot for r in risk_levels]
    cond_stability.append(row)

im2 = ax2.imshow(cond_stability, cmap="RdYlGn_r", aspect="auto", vmin=0, vmax=1)
ax2.set_xticks(range(len(risk_levels)))
ax2.set_xticklabels(risk_levels)
ax2.set_yticks(range(len(CONDITIONS)))
ax2.set_yticklabels(COND_SHORT)
for i in range(len(CONDITIONS)):
    for j in range(len(risk_levels)):
        val = cond_stability[i][j]
        if val > 0:
            ax2.text(j, i, f"{val:.0%}", ha="center", va="center",
                     fontsize=10, fontweight="bold",
                     color="white" if val > 0.6 else "black")
ax2.set_title("B) Per-Condition Verdict Stability\n(fraction of bootstrap resamples)", fontweight="bold")

fig.suptitle(f"CheXpert Bootstrap Stability ({n_boot} resamples)", fontsize=13, fontweight="bold")
plt.colorbar(im2, ax=ax2, label="Fraction", shrink=0.8)
plt.tight_layout()
plt.savefig(f"{OUT}/fig7_real_data_stability.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Summary table ---
print("\n=== Per-Method Stability (most frequent verdict / fraction) ===")
for m in METHODS:
    counts = {r: boot_verdicts[m].count(r) for r in risk_levels}
    dominant = max(counts, key=counts.get)
    frac = counts[dominant] / n_boot
    print(f"  {m:15s}  {dominant:8s}  {frac:.0%} consistent")

print("\n=== Per-Condition Stability ===")
for i, c in enumerate(CONDITIONS):
    counts = {r: boot_cond_verdicts[c].count(r) for r in risk_levels}
    dominant = max(counts, key=counts.get)
    frac = counts[dominant] / n_boot
    print(f"  {COND_SHORT[i]:12s}  {dominant:8s}  {frac:.0%} consistent")

print("\nKey takeaway: stable methods give the same answer regardless of")
print("which samples are included. The meta_classifier inherits stability")
print("from the reliable methods it trusts most (probe, statistical, hbac).")

---
## Recommendations

Based on synthetic benchmarks (Figs 1-5) and real CheXpert data (Fig 6):

1. **`meta_classifier`** -- Best overall (97.5% weighted accuracy). Trained on synthetic data
   with all 6 methods, it learns which methods to trust. Lowest false positive rate (3.3%)
   while catching weak shortcuts. On real data it is conservative -- only flags risk when
   the validated methods (probe, statistical, hbac) agree.
2. **`weighted_risk`** -- Best heuristic (71.7%). No trained model needed. Balances evidence
   across methods by weighting risk values. Moderate false positive rate.
3. **`indicator_count` / `majority_vote`** -- Simple counting. With 6 methods, they become
   too aggressive (100% FP rate) because noisy methods always produce some indicators.
   Better suited for 2-3 method setups.
4. **`multi_attribute`** -- Conservative on single-attribute data. Best when analyzing
   multiple demographic attributes simultaneously (race + sex + age).

**For papers:** report `meta_classifier` as primary, show `weighted_risk` as sensitivity
analysis. When methods disagree on real data (Fig 6), this disagreement is itself informative --
it means the evidence is ambiguous and warrants closer inspection.
